In [1]:
import pandas as pd      
import os                
import requests          
from datetime import datetime 

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)


In [2]:
DATASETS = {
    "covid_cases": {
        "url": "https://catalog.ourworldindata.org/garden/covid/latest/cases_deaths/cases_deaths.csv",
        "columns": [
            "country",
            "date",
            "new_cases",                        # новые случаи за день
            "total_cases",                      # всего случаев
            "new_deaths",                       # новые смерти за день
            "total_deaths",                     # всего смертей
            "new_cases_7_day_avg_right",        # сглаженные случаи (7 дней) — лучше для модели
            "new_deaths_7_day_avg_right",       # сглаженные смерти (7 дней)
            "new_cases_per_million",            # на миллион — для сравнения стран
            "new_deaths_per_million",           # на миллион
            "total_cases_per_million",
            "total_deaths_per_million",
            "weekly_cases",                     # недельные — для временных рядов
            "weekly_deaths",
            "weekly_pct_growth_cases",          # % роста за неделю — важная фича для модели
            "weekly_pct_growth_deaths",
            "cfr",                              # Case Fatality Rate — летальность
        ]
    },
    "covid_vaccinations": {
        "url": "https://catalog.ourworldindata.org/garden/covid/latest/vaccinations_global/vaccinations_global.csv",
        "columns": [
            "country",
            "date",
            "total_vaccinations_per_hundred",       # % населения получил хоть одну дозу
            "people_vaccinated_per_hundred",        # % хотя бы 1 доза
            "people_fully_vaccinated_per_hundred",  # % полностью вакцинированных
            "total_boosters_per_hundred",           # % получил бустер
            "daily_vaccinations_smoothed",          # темп вакцинации
            "people_unvaccinated",                  # абсолютное число невакцинированных
        ]
    },
    "covid_hospital": {
        "url": "https://catalog.ourworldindata.org/garden/covid/latest/hospital/hospital.csv",
        "columns": [
            "country",
            "date",
            "daily_occupancy_icu",              # занятых мест в реанимации
            "daily_occupancy_icu_per_1m",       # на миллион — для сравнения
            "daily_occupancy_hosp",             # занятых мест в больницах
            "daily_occupancy_hosp_per_1m",
            "weekly_admissions_hosp",           # новых госпитализаций за неделю
            "weekly_admissions_hosp_per_1m",
        ]
    },
    "mpox": {
        "url": "https://catalog.ourworldindata.org/garden/who/latest/monkeypox/monkeypox.csv",
        "columns": [
            "country",
            "date",
            "new_cases",
            "total_cases",
            "new_deaths",
            "total_deaths",
            "new_cases_smoothed",           # ← вот сглаженные в mpox называются так
            "new_deaths_smoothed",          # ← а не 7_day_avg_right как в covid
            "new_cases_per_million",
            "new_cases_smoothed_per_million",
            "new_deaths_per_million",
            "total_cases_per_million",
            "total_deaths_per_million",
        ]
    },
}

print("датасеты:")
for name, info in DATASETS.items():
    print(f"  • {name}: {len(info['columns'])} колонок")


датасеты:
  • covid_cases: 17 колонок
  • covid_vaccinations: 8 колонок
  • covid_hospital: 8 колонок
  • mpox: 13 колонок


In [ ]:
def download_and_filter(name, url, columns_needed):

    processed_path = f"data/processed/{name}.csv"
    raw_path = f"data/raw/{name}.csv"

    if os.path.exists(processed_path):
        df = pd.read_csv(processed_path)
        print(f" {name}: уже готов — {len(df):,} строк, {df.shape[1]} колонок")
        return df


    try:
        df_raw = pd.read_csv(url)
        df_raw.to_csv(raw_path, index=False) 
        available = [c for c in columns_needed if c in df_raw.columns]
        missing   = [c for c in columns_needed if c not in df_raw.columns]

        df_clean = df_raw[available].copy()
        df_clean.to_csv(processed_path, index=False)

        size_mb = os.path.getsize(processed_path) / (1024*1024)
        print(f"{name}: {len(df_clean):,} строк | {len(available)} колонок | {size_mb:.1f} MB")

        if missing:
            print(f"   Не найдены в файле: {missing}")

        return df_clean

    except Exception as e:
        print(f"{name}: {e}")
        return None



loaded = {}
for name, info in DATASETS.items():
    df = download_and_filter(name, info["url"], info["columns"])
    if df is not None:
        loaded[name] = df

print(f"\nЗагружено: {len(loaded)}/{len(DATASETS)} датасетов")


 Загружаю данные...

⚡ covid_cases: уже готов — 558,258 строк, 17 колонок
⚡ covid_vaccinations: уже готов — 203,057 строк, 8 колонок
⚡ covid_hospital: уже готов — 46,777 строк, 8 колонок
⚡ mpox: уже готов — 160,599 строк, 13 колонок

Загружено: 4/4 датасетов


In [4]:
pop = pd.read_csv(
    "https://raw.githubusercontent.com/datasets/population/master/data/population.csv"
)

pop_latest = pop.sort_values('Year').groupby('Country Name').last().reset_index()
pop_latest = pop_latest[['Country Name', 'Value']].rename(columns={
    'Country Name': 'country',
    'Value':        'population'
})

pop_latest.to_csv("data/processed/population.csv", index=False)
print(f"Population: {len(pop_latest)} countries saved")
print(pop_latest[pop_latest['country'] == 'Kazakhstan'])  

Population: 265 countries saved
        country  population
122  Kazakhstan    20592571


In [ ]:
from datetime import datetime

print("=" * 55)
print(" Файлы в data/processed/:")
print("=" * 55)

total_mb = 0
for f in sorted(os.listdir("data/processed")):
    if not f.endswith(".csv"):      # ← вот этой строки не хватало
        continue
    path = f"data/processed/{f}"
    mb = os.path.getsize(path) / (1024*1024)
    total_mb += mb
    df = pd.read_csv(path, nrows=0)
    print(f"  {f:<40} {mb:.1f} MB  |  {len(df.columns)} колонок")

print("=" * 55)
print(f"  {'ИТОГО':<40} {total_mb:.1f} MB")
print()
print(f" Слой 1 завершён: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("  Следующий шаг: biosecurity_layer2.ipynb (EDA)")

📁 Файлы в data/processed/:
  📄 covid_cases.csv                          79.2 MB  |  17 колонок
  📄 covid_hospital.csv                       2.3 MB  |  8 колонок
  📄 covid_vaccinations.csv                   14.6 MB  |  8 колонок
  📄 mpox.csv                                 10.9 MB  |  13 колонок
  📄 population.csv                           0.0 MB  |  2 колонок
  ИТОГО                                    107.0 MB

✅ Слой 1 завершён: 2026-04-19 23:59
➡️  Следующий шаг: biosecurity_layer2.ipynb (EDA)
